# Case RJ - Grupo 2

**Notebook para Exportação da Camada Gold em CSV (consumo no Power BI)**

Nomes dos Integrantes:
- Anna Cristina
- Gabriel Guimarães
- Juan Perri
- Milton Salgado
- Richard
- Yuri

## Sobre este notebook

O Databricks Free Edition não tem integração nativa com o Power BI (case, p. 9-10). Este notebook é o **último
passo do pipeline**: lê as tabelas da Gold já gravadas pelo `Silver_To_Gold.ipynb` e exporta CSVs para um
Volume do Unity Catalog, de onde o pacote é baixado e importado no Power BI Desktop.

- **Idempotente**: cada CSV é regravado por inteiro, com linhas em ordem fixa. Rodar de novo sobre a mesma Gold
  dá os mesmos arquivos. O manifesto traz um *checksum* por tabela para provar isso (soma de `xxhash64` das
  linhas, que não depende da ordem).
- **Rastreável**: o manifesto (`_manifesto.csv` e `_manifesto.json`) registra o id do pacote, a versão Delta de
  cada tabela de origem, linhas, colunas e checksum.
- **Sem dado pessoal**: o `id_cliente` só sai como hash (SHA-256 da Gold). Nome, e-mail e CPF não existem na Gold.
- **Duas pastas**:
  - `modelo/`: as 8 tabelas do modelo estrela de consumo (1 fato + 7 dimensões).
  - `analiticas/`: as 14 tabelas analíticas da Gold, para conferência e visuais rápidos.

### O que o modelo de consumo muda em relação à Gold

| Tabela | Ajuste | Motivo |
|---|---|---|
| `fato_atendimento` | Junta `qualidade_chamado_gold` por `id_chamado` (1 para 1, validado). Acrescenta `cliente_mapeado` e `recontato_elegivel` | Um único relacionamento no Power BI, sem ambiguidade |
| `dim_calendario` | Remove a linha `id_data = -1` | Permite inteligência temporal; chamados sem data continuam na fato |
| `dim_cliente` | Acrescenta o membro `SEM_CADASTRO` | Chamados com id sem cadastro não somem do relatório |

`recontato_elegivel` = tem id de cliente **e** motivo mapeado **e** abertura conhecida. Ter cadastro (`cliente_mapeado`)
é outra condição, de propósito.


In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
import csv, json, os, zipfile

CATALOGO = "dados_vcredit"
SCHEMA = "default"
VOLUME = "exportacao_powerbi"

# Decisão ainda em aberto no grupo (guia da Gold): nome real do atendente ou rótulo "Atendente <id>" no material
# do Comitê. O padrão mantém o comportamento atual da Gold (nome real). Mude no job via parâmetro.
dbutils.widgets.text("anonimizar_atendente", "false", "Anonimizar atendente (true/false)")
ANONIMIZAR_ATENDENTE = dbutils.widgets.get("anonimizar_atendente").strip().lower() == "true"

RAIZ = f"/Volumes/{CATALOGO}/{SCHEMA}/{VOLUME}"
PASTA_TMP = f"{RAIZ}/_tmp"
PACOTE_ID = datetime.now(timezone.utc).strftime("exp_%Y%m%d_%H%M%S")
SEM_CADASTRO = "SEM_CADASTRO"

TABELAS_MODELO = [
    "dim_calendario", "dim_hora", "dim_faixa_espera", "dim_canal", "dim_motivo",
    "dim_atendente", "dim_cliente", "fato_atendimento", "qualidade_chamado",
]
TABELAS_ANALITICAS = [
    "visao_geral", "kpi_mensal", "comparativo_yoy", "funil_atendimento", "etapa_mensal",
    "motivos_pareto", "motivo_etapa", "demanda_dia_hora", "atendentes_desempenho",
    "satisfacao_drivers", "perfil_cliente", "qualidade_dados", "impacto_indicadores",
    "simulacao_ganhos",
]

def nome_gold(nome):
    return f"{CATALOGO}.{SCHEMA}.{nome}_gold"

def gold(nome):
    return spark.table(nome_gold(nome))

def existe(nome):
    return spark.catalog.tableExists(nome_gold(nome))

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{SCHEMA}.{VOLUME}")
print(f"Pacote {PACOTE_ID} -> {RAIZ}")
print(f"Anonimizar atendente: {ANONIMIZAR_ATENDENTE}")

Pacote exp_20261022_012729 -> /Volumes/dados_vcredit/default/exportacao_powerbi
Anonimizar atendente: False


## Leitura da Gold, validações e modelo de consumo

Erros **bloqueiam** a exportação (chave duplicada, perda de chamado na junção, órfão fora do calendário).
A verificação de hora (`id_hora` x hora da abertura) só **informa**: ela vai para o manifesto, não derruba o job.

In [0]:
faltando = [n for n in TABELAS_MODELO if not existe(n)]
if faltando:
    raise ValueError(f"Tabelas Gold ausentes: {faltando}. Rode o Silver_To_Gold antes.")

def contar_e_checar_chave(df, chave, nome):
    n = df.count()
    distintas = df.select(chave).distinct().count()
    assert n == distintas, f"{nome}: chave {chave} repetida ({n} linhas, {distintas} distintas)"
    return n

fato = gold("fato_atendimento")
qual = gold("qualidade_chamado")
dim_cliente = gold("dim_cliente")

n_fato = contar_e_checar_chave(fato, "id_chamado", "fato_atendimento_gold")
n_qual = contar_e_checar_chave(qual, "id_chamado", "qualidade_chamado_gold")
assert n_fato == n_qual, f"fato ({n_fato}) e qualidade ({n_qual}) deveriam ter o mesmo número de chamados"
contar_e_checar_chave(dim_cliente, "id_cliente_hash", "dim_cliente_gold")

# --- dim_cliente: membro SEM_CADASTRO (hash fixo) para chamados com id sem cadastro ---
hash_sem_cadastro = spark.range(1).select(F.sha2(F.lit(SEM_CADASTRO), 256)).first()[0]
assert dim_cliente.where(F.col("id_cliente_hash") == hash_sem_cadastro).count() == 0, "SEM_CADASTRO já existe na dim_cliente_gold"
valores_sem = {"id_cliente_hash": hash_sem_cadastro, "faixa_etaria": "Sem cadastro", "regiao": "Sem cadastro"}
linha_sem = spark.range(1).select(
    *[F.lit(valores_sem.get(c.name)).cast(c.dataType).alias(c.name) for c in dim_cliente.schema.fields]
)
dim_cliente_bi = dim_cliente.unionByName(linha_sem)

# --- fato de consumo: fato + qualidade (1 para 1) + cliente_mapeado + recontato_elegivel ---
colunas_qualidade = [c for c in qual.columns if c != "id_chamado"]
clientes_cadastrados = dim_cliente.select("id_cliente_hash", F.lit(True).alias("cliente_mapeado"))

consumo = (
    fato.join(qual, "id_chamado", "inner")
    .join(clientes_cadastrados, "id_cliente_hash", "left")
    .withColumn("cliente_mapeado", F.coalesce(F.col("cliente_mapeado"), F.lit(False)))
    .withColumn(
        "id_cliente_hash",
        F.when(F.col("id_cliente_hash").isNotNull() & ~F.col("cliente_mapeado"), F.lit(hash_sem_cadastro))
         .otherwise(F.col("id_cliente_hash")),
    )
    .withColumn(
        "recontato_elegivel",
        (~F.col("sem_id_cliente")) & F.col("motivo_mapeado") & F.col("abertura_ts").isNotNull(),
    )
    .select(*fato.columns, *colunas_qualidade, "cliente_mapeado", "recontato_elegivel")
)
assert consumo.count() == n_fato, "A junção fato x qualidade perdeu ou multiplicou chamados"

# --- dimensões de consumo ---
def anonimizar(df):
    if not ANONIMIZAR_ATENDENTE:
        return df
    return df.withColumn(
        "rotulo_atendente",
        F.when(F.col("id_atendente") > 0, F.concat(F.lit("Atendente "), F.col("id_atendente").cast("string")))
         .otherwise(F.col("rotulo_atendente")),
    )

dim_calendario_bi = gold("dim_calendario").where(F.col("id_data") != -1)

# (arquivo, DataFrame, colunas de ordenação, tabelas Gold de origem)
MODELO_BI = [
    ("fato_atendimento", consumo, ["id_chamado"], ["fato_atendimento", "qualidade_chamado"]),
    ("dim_calendario", dim_calendario_bi, ["id_data"], ["dim_calendario"]),
    ("dim_hora", gold("dim_hora"), ["id_hora"], ["dim_hora"]),
    ("dim_faixa_espera", gold("dim_faixa_espera"), ["id_faixa_espera"], ["dim_faixa_espera"]),
    ("dim_canal", gold("dim_canal"), ["id_canal"], ["dim_canal"]),
    ("dim_motivo", gold("dim_motivo"), ["id_motivo"], ["dim_motivo"]),
    ("dim_atendente", anonimizar(gold("dim_atendente")), ["id_atendente"], ["dim_atendente"]),
    ("dim_cliente", dim_cliente_bi, ["id_cliente_hash"], ["dim_cliente"]),
]

# --- integridade referencial do modelo de consumo ---
chaves_fato = [
    ("id_hora", "dim_hora"), ("id_faixa_espera", "dim_faixa_espera"), ("id_canal", "dim_canal"),
    ("id_motivo", "dim_motivo"), ("id_atendente", "dim_atendente"),
]
dfs = {nome: df for nome, df, _, _ in MODELO_BI}
for fk, dim in chaves_fato:
    orfaos = consumo.join(dfs[dim].select(fk), fk, "left_anti").count()
    assert orfaos == 0, f"{orfaos} chamados com {fk} sem correspondência em {dim}"

orfaos_cliente = consumo.where(F.col("id_cliente_hash").isNotNull()).join(
    dim_cliente_bi.select("id_cliente_hash"), "id_cliente_hash", "left_anti").count()
assert orfaos_cliente == 0, f"{orfaos_cliente} chamados com id_cliente_hash fora da dim_cliente"

# id_data: só os chamados sem data (-1) ficam sem linha no calendário exportado
sem_data = consumo.where(F.col("id_data") == -1).count()
orfaos_data = consumo.join(dim_calendario_bi.select("id_data"), "id_data", "left_anti").count()
assert orfaos_data == sem_data, "Há chamados com data que não existem no calendário exportado"

# --- verificação informativa: hora do id_hora x hora de abertura_ts ---
desvio_hora = consumo.where(
    F.col("abertura_ts").isNotNull() & (F.col("id_hora") != -1) & (F.hour("abertura_ts") != F.col("id_hora"))
).count()

# --- conferência com as tabelas analíticas ---
if existe("visao_geral"):
    total_visao = gold("visao_geral").first()["total_chamados"]
    assert total_visao == n_fato, f"visao_geral_gold ({total_visao}) difere da fato ({n_fato})"
if existe("kpi_mensal"):
    total_kpi = gold("kpi_mensal").agg(F.sum("chamados")).first()[0]
    assert total_kpi == n_fato - sem_data, f"kpi_mensal_gold ({total_kpi}) difere da fato menos os sem data ({n_fato - sem_data})"

VERIFICACOES = {
    "chamados_fato": n_fato,
    "chamados_sem_data": sem_data,
    "chamados_id_cliente_sem_cadastro": consumo.where(F.col("id_cliente_hash").isNotNull() & ~F.col("cliente_mapeado")).count(),
    "chamados_recontato_elegivel": consumo.where("recontato_elegivel").count(),
    "informativo_hora_divergente": desvio_hora,
}
print(json.dumps(VERIFICACOES, indent=2, ensure_ascii=False))

{
  "chamados_fato": 131838,
  "chamados_sem_data": 0,
  "chamados_id_cliente_sem_cadastro": 1330,
  "chamados_recontato_elegivel": 131838,
  "informativo_hora_divergente": 0
}


## Exportação

Uma função só, chamada uma vez por tabela. O Spark grava a pasta `part-*.csv` em `_tmp`; ela é copiada para o nome
final (`<tabela>.csv`) e a pasta temporária é apagada.

Formato (pensado para o Power BI): UTF-8, vírgula, cabeçalho, aspas duplas, ponto decimal, datas `yyyy-MM-dd`,
timestamps `yyyy-MM-dd HH:mm:ss`, booleanos `true`/`false`, nulo = campo vazio.

In [0]:
def resumo(df):
    """Linhas, colunas e checksum independente da ordem (soma de xxhash64 por linha, em decimal para não estourar)."""
    cols = [F.col(c) for c in df.columns]
    r = df.agg(
        F.count(F.lit(1)).alias("n"),
        F.sum(F.xxhash64(*cols).cast("decimal(38,0)")).alias("ck"),
    ).first()
    return int(r["n"]), (str(r["ck"]) if r["ck"] is not None else "0")

def versao_delta(nome):
    try:
        h = spark.sql(f"DESCRIBE HISTORY {nome_gold(nome)} LIMIT 1").first()
        return int(h["version"]), str(h["timestamp"])
    except Exception:
        return None, None

def exportar(df, subpasta, nome, ordem):
    pasta = f"{RAIZ}/{subpasta}"
    tmp = f"{PASTA_TMP}/{nome}"
    final = f"{pasta}/{nome}.csv"
    dbutils.fs.mkdirs(pasta)
    dbutils.fs.rm(tmp, True)
    (
        df.orderBy(*ordem).coalesce(1).write.mode("overwrite")
        .option("header", "true").option("encoding", "UTF-8")
        .option("sep", ",").option("quote", '"').option("escape", '"')
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss").option("dateFormat", "yyyy-MM-dd")
        .option("nullValue", "").option("emptyValue", "")
        .csv(tmp)
    )
    partes = [f.path for f in dbutils.fs.ls(tmp) if f.name.startswith("part-") and f.name.endswith(".csv")]
    assert len(partes) == 1, f"{nome}: esperava 1 arquivo part-*.csv, achei {len(partes)}"
    dbutils.fs.rm(final)
    dbutils.fs.cp(partes[0], final)
    dbutils.fs.rm(tmp, True)

    linhas, checksum = resumo(df)
    lidas = (
        spark.read.option("header", "true").option("multiLine", "true").option("escape", '"').csv(final).count()
    )
    assert lidas == linhas, f"{nome}: o CSV tem {lidas} linhas, a tabela tem {linhas}"
    return final, linhas, len(df.columns), checksum

In [0]:
exportado_em = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
manifesto = []

def registrar(grupo, nome, df, ordem, fontes):
    caminho, linhas, colunas, checksum = exportar(df, grupo, nome, ordem)
    v = versao_delta(fontes[0])
    manifesto.append({
        "pacote_id": PACOTE_ID, "exportado_em_utc": exportado_em, "grupo": grupo,
        "arquivo": f"{grupo}/{nome}.csv", "tabelas_gold_origem": "+".join(f"{f}_gold" for f in fontes),
        "linhas": linhas, "colunas": colunas, "checksum_linhas": checksum,
        "versao_delta_origem": v[0], "timestamp_delta_origem": v[1],
        "anonimizar_atendente": ANONIMIZAR_ATENDENTE,
    })
    print(f"{grupo}/{nome}.csv: {linhas:,} linhas, {colunas} colunas")

for nome, df, ordem, fontes in MODELO_BI:
    registrar("modelo", nome, df, ordem, fontes)

for nome in TABELAS_ANALITICAS:
    if not existe(nome):
        print(f"analiticas/{nome}: tabela Gold não existe, ignorada")
        continue
    df = gold(nome)
    if nome == "atendentes_desempenho":
        df = anonimizar(df)
    registrar("analiticas", nome, df, df.columns, [nome])

modelo/fato_atendimento.csv: 131,838 linhas, 47 colunas
modelo/dim_calendario.csv: 730 linhas, 12 colunas
modelo/dim_hora.csv: 25 linhas, 4 colunas
modelo/dim_faixa_espera.csv: 6 linhas, 3 colunas
modelo/dim_canal.csv: 5 linhas, 6 colunas
modelo/dim_motivo.csv: 15 linhas, 5 colunas
modelo/dim_atendente.csv: 28 linhas, 7 colunas
modelo/dim_cliente.csv: 47,706 linhas, 5 colunas
analiticas/visao_geral.csv: 1 linhas, 16 colunas
analiticas/kpi_mensal.csv: 19 linhas, 16 colunas
analiticas/comparativo_yoy.csv: 12 linhas, 14 colunas
analiticas/funil_atendimento.csv: 4 linhas, 15 colunas
analiticas/etapa_mensal.csv: 57 linhas, 9 colunas
analiticas/motivos_pareto.csv: 14 linhas, 15 colunas
analiticas/motivo_etapa.csv: 42 linhas, 8 colunas
analiticas/demanda_dia_hora.csv: 168 linhas, 8 colunas
analiticas/atendentes_desempenho.csv: 25 linhas, 20 colunas
analiticas/satisfacao_drivers.csv: 28 linhas, 5 colunas
analiticas/perfil_cliente.csv: 36 linhas, 9 colunas
analiticas/qualidade_dados.csv: 27 lin

## Manifesto e pacote

O manifesto fica em CSV (para abrir no Power BI) e em JSON (com as verificações). O pacote `.zip` junta tudo
para baixar de uma vez: **Catálogo → `dados_vcredit` → `default` → Volumes → `exportacao_powerbi` → `pacote_powerbi.zip` → Baixar**.

In [0]:
import io

# Tudo é montado em memória e gravado direto no Volume (o serverless não permite acessar /tmp via dbutils).
campos = list(manifesto[0].keys())
buf = io.StringIO()
w = csv.DictWriter(buf, fieldnames=campos)
w.writeheader()
w.writerows(manifesto)
txt_csv = buf.getvalue()
txt_json = json.dumps({"pacote_id": PACOTE_ID, "exportado_em_utc": exportado_em,
                       "verificacoes": VERIFICACOES, "arquivos": manifesto}, ensure_ascii=False, indent=2)

for nome_arq, conteudo in [("_manifesto.csv", txt_csv), ("_manifesto.json", txt_json)]:
    with open(f"{RAIZ}/{nome_arq}", "w", encoding="utf-8", newline="") as f:
        f.write(conteudo)

# --- pacote zip (montado em memória; o Volume não aceita escrita aleatória) ---
mem = io.BytesIO()
with zipfile.ZipFile(mem, "w", zipfile.ZIP_DEFLATED) as z:
    for item in manifesto:
        with open(f"{RAIZ}/{item['arquivo']}", "rb") as f:
            z.writestr(item["arquivo"], f.read())
    z.writestr("_manifesto.csv", txt_csv.encode("utf-8"))
    z.writestr("_manifesto.json", txt_json.encode("utf-8"))

with open(f"{RAIZ}/pacote_powerbi.zip", "wb") as f:
    f.write(mem.getvalue())
dbutils.fs.rm(PASTA_TMP, True)

total_modelo = sum(1 for m in manifesto if m["grupo"] == "modelo")
total_analiticas = sum(1 for m in manifesto if m["grupo"] == "analiticas")
print(f"Pacote {PACOTE_ID}: {total_modelo} tabelas do modelo + {total_analiticas} analíticas")
print(f"Baixe: {RAIZ}/pacote_powerbi.zip")

Pacote exp_20261022_012729: 8 tabelas do modelo + 14 analíticas
Baixe: /Volumes/dados_vcredit/default/exportacao_powerbi/pacote_powerbi.zip


## Checklist antes de aplicar uma mudança

- [ ] O `Silver_To_Gold.ipynb` rodou sem erro antes deste notebook.
- [ ] As contagens do manifesto batem com o resumo da Gold (fato: mesmo número de chamados da Silver).
- [ ] Rodamos duas vezes e os `checksum_linhas` do manifesto são iguais (idempotente).
- [ ] Nenhuma célula exibe `id_cliente` em claro (a Gold só tem o hash).
- [ ] Decisão do grupo registrada: `anonimizar_atendente` = true ou false no material do Comitê.